In [ ]:

# 用户输入参数：只改这个单元即可。
config = {
    # 输入源与输出目录配置。
    "input_config": {
        "starrocks_connection": {
            "host": "10.12.105.139",  # StarRocks 地址
            "port": 30030,  # StarRocks 端口
            "user": "",  # StarRocks 用户名
            "password": "",  # StarRocks 密码
            "database": "ads",  # StarRocks 数据库名
            "charset": "utf8mb4",
        },
        "starrocks_table": "ads.ads_meta_unified_unique_meta_data_acc_d",  # metadata 表名
        "prod_output_root": "s3://scibase-service/agentic-search/raw-content/v4",  # 正式输出根目录
        "test_output_root": "s3://web-parse-hw60p/linfeng/agentic_search/oa_chunk_data_test/v11_test",  # 测试输出根目录
        "test_target_sha256s": [  # test 模式下可固定跑指定样本
            "01d845845d1d8207db5432c93f5261e847710fc91423ff7b34195f791be67445",
        ],
        # StarRocks 查询模板：如需改字段、过滤条件或排序逻辑，改这里。
        "sql_templates": {
            # 主查询：正常拉取一批待处理 metadata 明细。
            "fetch_meta_base": """
                SELECT metadata_type,
                       doi,
                       isbn13,
                       title,
                       author,
                       language,
                       access_xinghe_repository_origin_url AS origin_url,
                       access_xinghe_repository_origin_path AS origin_path,
                       access_xinghe_repository_processed_path AS processed_path,
                       lower(trim(access_xinghe_repository_sha256)) AS source_sha256,
                       access_xinghe_repository_model_name AS model_name,
                       access_xinghe_repository_model_version AS model_version,
                       access_is_oa,
                       abstract
                FROM {starrocks_table}
                WHERE dt = '2026-06-22'
                  AND access_xinghe_repository_process_status = 1
                  AND access_xinghe_repository_processed_path IS NOT NULL
                  AND access_xinghe_repository_processed_path != ''
                  AND access_xinghe_repository_sha256 IS NOT NULL
                  AND trim(access_xinghe_repository_sha256) != ''
                  {bucket_clause}
                  {last_clause}
                ORDER BY lower(trim(access_xinghe_repository_sha256))
                LIMIT {limit_rows}
            """,
            # test 模式补样本：把 test_target_sha256s 里点名的 sha256 额外查出来。
            "fetch_meta_test_target_sha256s": """
                SELECT metadata_type,
                       doi,
                       isbn13,
                       title,
                       author,
                       language,
                       access_xinghe_repository_origin_url AS origin_url,
                       access_xinghe_repository_origin_path AS origin_path,
                       access_xinghe_repository_processed_path AS processed_path,
                       lower(trim(access_xinghe_repository_sha256)) AS source_sha256,
                       access_xinghe_repository_model_name AS model_name,
                       access_xinghe_repository_model_version AS model_version,
                       access_is_oa,
                       abstract
                FROM {starrocks_table}
                WHERE dt = '2026-06-22'
                  AND access_xinghe_repository_process_status = 1
                  AND access_xinghe_repository_processed_path IS NOT NULL
                  AND access_xinghe_repository_processed_path != ''
                  AND access_xinghe_repository_sha256 IS NOT NULL
                  AND trim(access_xinghe_repository_sha256) != ''
                  {bucket_clause}
                  AND lower(trim(access_xinghe_repository_sha256)) IN ({quoted_sha256s})
            """,
            # 统计查询：只算剩余待处理文档数，不返回明细。
            "count_pending_rows": """
                SELECT COUNT(DISTINCT lower(trim(access_xinghe_repository_sha256))) AS total_count
                FROM {starrocks_table}
                WHERE dt = '2026-06-22'
                  AND access_xinghe_repository_process_status = 1
                  AND access_xinghe_repository_processed_path IS NOT NULL
                  AND access_xinghe_repository_processed_path != ''
                  AND access_xinghe_repository_sha256 IS NOT NULL
                  AND trim(access_xinghe_repository_sha256) != ''
                  {bucket_clause}
                  {last_clause}
            """,
        },
    },
    # 运行开关与吞吐参数。
    "runtime_config": {
        "run_mode": "prod",  # "prod" 全量模式；"test" 小样本模式
        "run_main_flow": False,  # 是否执行主流程：拉 metadata、读取原文并写出 doc.json
        "run_slow_path": True,  # 是否继续处理慢路径队列：补跑大文件 / 慢文件
        "test_row_limit": 20,  # test 模式单批/总样本上限
        "overwrite_existing_docs": False,  # True 时覆盖已存在 doc.json
        "preview_rows": 5,  # 预览下一批 metadata 条数
        "enable_driver_path_existence_check": False,  # 主流程是否先在 driver 侧判断输出是否已存在
        "disable_boto_response_checksum_validation": True,  # 兼容部分对象存储响应校验
        "batch_size_prod": 140000,  # prod 模式每批 metadata 行数
        "batch_size_test": 20,  # test 模式每批 metadata 行数
        "rows_per_partition_prod": 500,  # prod 模式目标每分区行数
        "rows_per_partition_test": 1,  # test 模式目标每分区行数
        "target_batch_partitions_prod": 512,  # prod 模式目标分区数
        "target_batch_partitions_test": 20,  # test 模式目标分区数
        "max_batch_partitions_prod": 1024,  # prod 模式最大分区数
        "max_batch_partitions_test": 20,  # test 模式最大分区数
        "sha_bucket_count_prod": 256,  # prod 模式 sha bucket 数
        "sha_bucket_count_test": 8,  # test 模式 sha bucket 数
        "max_s3_connections_prod": 64,  # prod 模式 boto 最大连接数
        "max_s3_connections_test": 16,  # test 模式 boto 最大连接数
        "target_global_read_concurrency_prod": 512,  # prod 模式目标全局读并发
        "target_global_read_concurrency_test": 16,  # test 模式目标全局读并发
        "target_global_write_concurrency_prod": 1024,  # prod 模式目标全局写并发
        "target_global_write_concurrency_test": 16,  # test 模式目标全局写并发
        "slow_path_bytes_threshold_prod": 50 * 1024 * 1024,  # prod 模式慢路径字节阈值
        "slow_path_bytes_threshold_test": 10 * 1024 * 1024,  # test 模式慢路径字节阈值
        "oversized_bytes_threshold_prod": 512 * 1024 * 1024,  # prod 模式超大文档阈值
        "oversized_bytes_threshold_test": 128 * 1024 * 1024,  # test 模式超大文档阈值
        "slow_path_batch_size_prod": 500,  # prod 模式慢路径每批行数
        "slow_path_batch_size_test": 20,  # test 模式慢路径每批行数
        "slow_path_read_threads": None,  # 慢路径读线程数，None 表示自动
        "slow_path_write_threads": None,  # 慢路径写线程数，None 表示自动
        "slow_path_force_driver_exists_check": True,  # 慢路径是否强制先检查目标文件是否已存在
        "driver_write_max_pending_rows_prod": 256,  # prod 模式触发 driver 直写的最大待处理行数
        "driver_write_max_pending_rows_test": 16,  # test 模式触发 driver 直写的最大待处理行数
        "driver_write_max_read_threads_prod": 16,  # prod 模式 driver 直写最大读线程数
        "driver_write_max_read_threads_test": 4,  # test 模式 driver 直写最大读线程数
        "driver_write_max_write_threads_prod": 32,  # prod 模式 driver 直写最大写线程数
        "driver_write_max_write_threads_test": 8,  # test 模式 driver 直写最大写线程数
    },
}


In [ ]:

import bz2
from collections import defaultdict
import gzip
import hashlib
import io
import json
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import uuid
from urllib.parse import urlsplit, urlunsplit

import boto3
import pymysql
from botocore.client import Config as BotocoreConfig
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F, types as T

from app.common.json_util import *
from xinghe.spark import *
from xinghe.s3 import *
from xinghe.s3 import S3LineWriter, delete_s3_object, put_s3_object, read_s3_object, read_s3_row, read_s3_rows

input_config = config["input_config"]
runtime_config = config["runtime_config"]

STARROCKS_CONNECTION = input_config["starrocks_connection"]
STARROCKS_TABLE = input_config["starrocks_table"]
RUN_MODE = runtime_config["run_mode"]
RUN_MAIN_FLOW = runtime_config["run_main_flow"]
RUN_SLOW_PATH = runtime_config["run_slow_path"]
PROD_OUTPUT_ROOT = input_config["prod_output_root"]
TEST_OUTPUT_ROOT = input_config["test_output_root"]
TEST_ROW_LIMIT = int(runtime_config["test_row_limit"])
TEST_TARGET_SHA256S = input_config["test_target_sha256s"]
SQL_TEMPLATES = input_config["sql_templates"]
OVERWRITE_EXISTING_DOCS = runtime_config["overwrite_existing_docs"]
PREVIEW_ROWS = int(runtime_config["preview_rows"])
ENABLE_DRIVER_PATH_EXISTENCE_CHECK = runtime_config["enable_driver_path_existence_check"]
DISABLE_BOTO_RESPONSE_CHECKSUM_VALIDATION = runtime_config["disable_boto_response_checksum_validation"]

if RUN_MODE not in {"prod", "test"}:
    raise ValueError(f"RUN_MODE 只支持 'prod' 或 'test'，当前值: {RUN_MODE!r}")


def pick_mode_value(prod_key, test_key):
    return runtime_config[prod_key] if RUN_MODE == "prod" else runtime_config[test_key]


OUTPUT_ROOT = PROD_OUTPUT_ROOT if RUN_MODE == "prod" else TEST_OUTPUT_ROOT
OA_ROOT = f"{OUTPUT_ROOT}/oa"
OTHERS_ROOT = f"{OUTPUT_ROOT}/others"
SUMMARY_PATH = f"{OUTPUT_ROOT}/summary.json"
ROW_PROGRESS_ROOT = f"{OUTPUT_ROOT}/_row_progress".replace("s3://", "s3a://", 1)
BUCKET_PROGRESS_ROOT = f"{OUTPUT_ROOT}/_bucket_progress"

BATCH_SIZE = int(pick_mode_value("batch_size_prod", "batch_size_test"))
ROWS_PER_PARTITION = int(pick_mode_value("rows_per_partition_prod", "rows_per_partition_test"))
TARGET_BATCH_PARTITIONS = int(pick_mode_value("target_batch_partitions_prod", "target_batch_partitions_test"))
MAX_BATCH_PARTITIONS = int(pick_mode_value("max_batch_partitions_prod", "max_batch_partitions_test"))
SHA_BUCKET_COUNT = int(pick_mode_value("sha_bucket_count_prod", "sha_bucket_count_test"))
MAX_S3_CONNECTIONS = int(pick_mode_value("max_s3_connections_prod", "max_s3_connections_test"))
TARGET_GLOBAL_READ_CONCURRENCY = int(pick_mode_value("target_global_read_concurrency_prod", "target_global_read_concurrency_test"))
TARGET_GLOBAL_WRITE_CONCURRENCY = int(pick_mode_value("target_global_write_concurrency_prod", "target_global_write_concurrency_test"))
SLOW_PATH_QUEUE_ROOT = f"{OUTPUT_ROOT}/_slow_path_queue"
SLOW_PATH_BYTES_THRESHOLD = int(pick_mode_value("slow_path_bytes_threshold_prod", "slow_path_bytes_threshold_test"))
OVERSIZED_PATH_QUEUE_ROOT = f"{OUTPUT_ROOT}/_oversized_path_queue"
OVERSIZED_BYTES_THRESHOLD = int(pick_mode_value("oversized_bytes_threshold_prod", "oversized_bytes_threshold_test"))
SLOW_PATH_BATCH_SIZE = int(pick_mode_value("slow_path_batch_size_prod", "slow_path_batch_size_test"))
SLOW_PATH_QUEUE_SHARD_ROWS = SLOW_PATH_BATCH_SIZE
SLOW_PATH_TARGET_PARTITIONS = min(TARGET_BATCH_PARTITIONS, 32) if RUN_MODE == "prod" else 4
SLOW_PATH_READ_THREADS = runtime_config["slow_path_read_threads"]
SLOW_PATH_WRITE_THREADS = runtime_config["slow_path_write_threads"]
SLOW_PATH_FORCE_DRIVER_EXISTS_CHECK = runtime_config["slow_path_force_driver_exists_check"]
DRIVER_WRITE_MAX_PENDING_ROWS = int(pick_mode_value("driver_write_max_pending_rows_prod", "driver_write_max_pending_rows_test"))
DRIVER_WRITE_MAX_READ_THREADS = min(MAX_S3_CONNECTIONS, int(pick_mode_value("driver_write_max_read_threads_prod", "driver_write_max_read_threads_test")))
DRIVER_WRITE_MAX_WRITE_THREADS = min(MAX_S3_CONNECTIONS, int(pick_mode_value("driver_write_max_write_threads_prod", "driver_write_max_write_threads_test")))

print(f"RUN_MODE={RUN_MODE}")
print(f"RUN_MAIN_FLOW={RUN_MAIN_FLOW}")
print(f"RUN_SLOW_PATH={RUN_SLOW_PATH}")
print(f"BATCH_SIZE={BATCH_SIZE}")
print(f"OVERWRITE_EXISTING_DOCS={OVERWRITE_EXISTING_DOCS}")
print(f"ROWS_PER_PARTITION={ROWS_PER_PARTITION}")
print(f"TARGET_BATCH_PARTITIONS={TARGET_BATCH_PARTITIONS}")
print(f"MAX_BATCH_PARTITIONS={MAX_BATCH_PARTITIONS}")
print(f"MAX_S3_CONNECTIONS={MAX_S3_CONNECTIONS}")
print(f"SHA_BUCKET_COUNT={SHA_BUCKET_COUNT}")
print(f"TARGET_GLOBAL_READ_CONCURRENCY={TARGET_GLOBAL_READ_CONCURRENCY}")
print(f"TARGET_GLOBAL_WRITE_CONCURRENCY={TARGET_GLOBAL_WRITE_CONCURRENCY}")
print(f"SLOW_PATH_QUEUE_ROOT={SLOW_PATH_QUEUE_ROOT}")
print(f"SLOW_PATH_BYTES_THRESHOLD={SLOW_PATH_BYTES_THRESHOLD}")
print(f"OVERSIZED_PATH_QUEUE_ROOT={OVERSIZED_PATH_QUEUE_ROOT}")
print(f"OVERSIZED_BYTES_THRESHOLD={OVERSIZED_BYTES_THRESHOLD}")
print(f"SLOW_PATH_BATCH_SIZE={SLOW_PATH_BATCH_SIZE}")
print(f"SLOW_PATH_QUEUE_SHARD_ROWS={SLOW_PATH_QUEUE_SHARD_ROWS}")
print(f"SLOW_PATH_TARGET_PARTITIONS={SLOW_PATH_TARGET_PARTITIONS}")
print(f"SLOW_PATH_READ_THREADS={SLOW_PATH_READ_THREADS}")
print(f"SLOW_PATH_WRITE_THREADS={SLOW_PATH_WRITE_THREADS}")
print(f"SLOW_PATH_FORCE_DRIVER_EXISTS_CHECK={SLOW_PATH_FORCE_DRIVER_EXISTS_CHECK}")
print(f"DRIVER_WRITE_MAX_PENDING_ROWS={DRIVER_WRITE_MAX_PENDING_ROWS}")
print(f"DRIVER_WRITE_MAX_READ_THREADS={DRIVER_WRITE_MAX_READ_THREADS}")
print(f"DRIVER_WRITE_MAX_WRITE_THREADS={DRIVER_WRITE_MAX_WRITE_THREADS}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")
print(f"ROW_PROGRESS_ROOT={ROW_PROGRESS_ROOT}")
print(f"BUCKET_PROGRESS_ROOT={BUCKET_PROGRESS_ROOT}")
print(f"ENABLE_DRIVER_PATH_EXISTENCE_CHECK={ENABLE_DRIVER_PATH_EXISTENCE_CHECK}")
print(f"DISABLE_BOTO_RESPONSE_CHECKSUM_VALIDATION={DISABLE_BOTO_RESPONSE_CHECKSUM_VALIDATION}")
print(f"TEST_TARGET_SHA256S={TEST_TARGET_SHA256S}")
REAL_DOC_FIELDS = [
    "track_id", "sha256", "process_status", "processed_path", "origin_url", "origin_path",
    "file_format", "file_type", "content_type", "content_length", "page_cnt", "is_broken",
    "obtain_timestamp", "language", "title", "author", "abstract", "category", "major",
    "major_2", "major_3", "source", "db_source", "subject", "doi", "keyword", "isbn",
    "issn", "issn_p", "issn_e", "magazine", "pub_time", "volume", "area", "grade_class",
    "grade", "data_date", "dt", "start_ts", "content_list", "ali_pdf_path", "theme",
    "sub_path", "page_index_range", "content", "lang", "model_name", "model_version",
    "model_process_ts", "pred_major", "pred_major2", "pred_major3", "ext", "fail_msg",
    "end_ts", "cost_sec", "id", "doc_loc", "doc_id"
]

ROW_PROGRESS_SCHEMA = T.StructType([
    T.StructField("source_sha256", T.StringType(), True),
    T.StructField("processed_path", T.StringType(), True),
    T.StructField("access_is_oa", T.StringType(), True),
    T.StructField("batch_index", T.LongType(), True),
    T.StructField("completed_at", T.StringType(), True),
])

print(f"RUN_MODE={RUN_MODE}")
print(f"BATCH_SIZE={BATCH_SIZE}")
print(f"OVERWRITE_EXISTING_DOCS={OVERWRITE_EXISTING_DOCS}")
print(f"ROWS_PER_PARTITION={ROWS_PER_PARTITION}")
print(f"TARGET_BATCH_PARTITIONS={TARGET_BATCH_PARTITIONS}")
print(f"MAX_BATCH_PARTITIONS={MAX_BATCH_PARTITIONS}")
print(f"MAX_S3_CONNECTIONS={MAX_S3_CONNECTIONS}")
print(f"SHA_BUCKET_COUNT={SHA_BUCKET_COUNT}")
print(f"TARGET_GLOBAL_READ_CONCURRENCY={TARGET_GLOBAL_READ_CONCURRENCY}")
print(f"TARGET_GLOBAL_WRITE_CONCURRENCY={TARGET_GLOBAL_WRITE_CONCURRENCY}")
print(f"SLOW_PATH_QUEUE_ROOT={SLOW_PATH_QUEUE_ROOT}")
print(f"SLOW_PATH_BYTES_THRESHOLD={SLOW_PATH_BYTES_THRESHOLD}")
print(f"OVERSIZED_PATH_QUEUE_ROOT={OVERSIZED_PATH_QUEUE_ROOT}")
print(f"OVERSIZED_BYTES_THRESHOLD={OVERSIZED_BYTES_THRESHOLD}")
print(f"SLOW_PATH_BATCH_SIZE={SLOW_PATH_BATCH_SIZE}")
print(f"SLOW_PATH_QUEUE_SHARD_ROWS={SLOW_PATH_QUEUE_SHARD_ROWS}")
print(f"SLOW_PATH_TARGET_PARTITIONS={SLOW_PATH_TARGET_PARTITIONS}")
print(f"SLOW_PATH_READ_THREADS={SLOW_PATH_READ_THREADS}")
print(f"SLOW_PATH_WRITE_THREADS={SLOW_PATH_WRITE_THREADS}")
print(f"SLOW_PATH_FORCE_DRIVER_EXISTS_CHECK={SLOW_PATH_FORCE_DRIVER_EXISTS_CHECK}")
print(f"DRIVER_WRITE_MAX_PENDING_ROWS={DRIVER_WRITE_MAX_PENDING_ROWS}")
print(f"DRIVER_WRITE_MAX_READ_THREADS={DRIVER_WRITE_MAX_READ_THREADS}")
print(f"DRIVER_WRITE_MAX_WRITE_THREADS={DRIVER_WRITE_MAX_WRITE_THREADS}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")
print(f"ROW_PROGRESS_ROOT={ROW_PROGRESS_ROOT}")
print(f"BUCKET_PROGRESS_ROOT={BUCKET_PROGRESS_ROOT}")
print(f"ENABLE_DRIVER_PATH_EXISTENCE_CHECK={ENABLE_DRIVER_PATH_EXISTENCE_CHECK}")
print(f"DISABLE_BOTO_RESPONSE_CHECKSUM_VALIDATION={DISABLE_BOTO_RESPONSE_CHECKSUM_VALIDATION}")
print(f"TEST_TARGET_SHA256S={TEST_TARGET_SHA256S}")


In [ ]:

# Spark 初始化：如需调 executor / queue / driver 内存，只改这个单元。
spark_config = {
    "app_name": "houlinfeng_starrocks_data",  # Spark app 名称
    "spark_conf": {
        "spark_conf_name": "spark_4",  # 使用的 Spark 资源模板
        "skip_success_check": True,
        "spark.driver.memory": "8g",  # driver 内存
        "spark.driver.maxResultSize": "2g",  # driver 最大结果集
        "spark.ui.enabled": "false",
        "spark.ui.retainedJobs": "20",
        "spark.ui.retainedStages": "20",
        "spark.ui.retainedTasks": "1000",
        "spark.sql.ui.retainedExecutions": "20",
        "spark.speculation": "true",  # 开启 speculative execution
        "spark.speculation.multiplier": "1.5",
        "spark.speculation.quantile": "0.90",
        "spark.speculation.minTaskRuntime": "60000",
        # "spark.yarn.queue": "root.warehouse",  # 如需指定队列可打开
        # "spark.sql.shuffle.partitions": 1000,  # 如需固定 shuffle 分区数可打开
    },
}

spark = new_spark_session(spark_config["app_name"], spark_config["spark_conf"])
spark.sparkContext.setLogLevel("ERROR")
spark


In [ ]:
def ensure_s3a_path(path):
    if path.startswith("s3://"):
        return "s3a://" + path[len("s3://"):]
    return path


def strip_bytes_suffix(path):
    parts = urlsplit(path)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))


def normalize_bool_text(value):
    return "true" if str(value).strip().lower() == "true" else "false"


def quote_sql(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def parse_bytes_length_from_processed_path(processed_path):
    text = str(processed_path or "").strip()
    if "?bytes=" not in text:
        return None
    try:
        _, bytes_part = text.split("?bytes=", 1)
        _, length_str = bytes_part.split(",", 1)
        return int(length_str)
    except Exception:
        return None


def classify_processed_path(processed_path):
    text = str(processed_path or "").strip()
    if "?bytes=" in text:
        byte_len = parse_bytes_length_from_processed_path(text)
        if byte_len is None:
            return "bytes"
        return f"bytes:{int(byte_len / (1024 * 1024))}MB"
    if text.endswith(".jsonl.gz"):
        return "jsonl.gz"
    if text.endswith(".jsonl.bz2"):
        return "jsonl.bz2"
    if text.endswith(".jsonl"):
        return "jsonl"
    if text.endswith(".gz"):
        return "gz"
    if text.endswith(".bz2"):
        return "bz2"
    return "other"


def summarize_processed_path_types(meta_rows):
    summary = defaultdict(int)
    for meta_row in meta_rows:
        summary[classify_processed_path(meta_row.get("processed_path"))] += 1
    return dict(sorted(summary.items(), key=lambda item: (-item[1], item[0])))


def summarize_result_rows(result_rows, top_n=5):
    top_n = max(1, int(top_n or 1))
    written_rows = [row for row in result_rows if row.get("status") == "written"]
    sorted_rows = sorted(
        written_rows,
        key=lambda row: (
            float(row.get("read_seconds") or 0.0) + float(row.get("write_seconds") or 0.0),
            float(row.get("read_seconds") or 0.0),
            float(row.get("write_seconds") or 0.0),
        ),
        reverse=True,
    )
    return [
        {
            "source_sha256": row.get("source_sha256"),
            "processed_path": row.get("processed_path"),
            "path_type": classify_processed_path(row.get("processed_path")),
            "read_seconds": float(row.get("read_seconds") or 0.0),
            "write_seconds": float(row.get("write_seconds") or 0.0),
            "total_seconds": float(row.get("read_seconds") or 0.0) + float(row.get("write_seconds") or 0.0),
        }
        for row in sorted_rows[:top_n]
    ]


def should_scan_processed_path_by_sha(processed_path):
    text = str(processed_path or "").strip()
    return ("?bytes=" not in text) and text.endswith((".jsonl", ".jsonl.gz", ".jsonl.bz2"))


def build_boto_s3_config(disable_response_checksum_validation=False):
    config_kwargs = {
        "s3": {"addressing_style": "path"},
        "retries": {"max_attempts": 8, "mode": "standard"},
        "connect_timeout": 600,
        "read_timeout": 600,
        "max_pool_connections": MAX_S3_CONNECTIONS,
    }
    if disable_response_checksum_validation:
        config_kwargs["request_checksum_calculation"] = "when_required"
        config_kwargs["response_checksum_validation"] = "when_required"
    try:
        return BotocoreConfig(**config_kwargs)
    except TypeError:
        config_kwargs.pop("request_checksum_calculation", None)
        config_kwargs.pop("response_checksum_validation", None)
        try:
            return BotocoreConfig(**config_kwargs)
        except TypeError:
            config_kwargs["retries"] = {"max_attempts": 8}
            return BotocoreConfig(**config_kwargs)


def get_boto_s3_client(path, outside=False, disable_response_checksum_validation=False):
    s3_config = get_s3_config(path, outside)
    return boto3.client(
        "s3",
        aws_access_key_id=s3_config["ak"],
        aws_secret_access_key=s3_config["sk"],
        endpoint_url=s3_config["endpoint"],
        config=build_boto_s3_config(disable_response_checksum_validation=disable_response_checksum_validation),
    )


class ClientCache:
    def __init__(self, disable_response_checksum_validation=DISABLE_BOTO_RESPONSE_CHECKSUM_VALIDATION):
        self.cache = {}
        self.disable_response_checksum_validation = disable_response_checksum_validation

    def get_client(self, path):
        bucket, _ = split_s3_path(path)
        cache_key = (bucket, self.disable_response_checksum_validation)
        if not bucket:
            return get_boto_s3_client(
                path,
                disable_response_checksum_validation=self.disable_response_checksum_validation,
            )
        if cache_key not in self.cache:
            self.cache[cache_key] = get_boto_s3_client(
                path,
                disable_response_checksum_validation=self.disable_response_checksum_validation,
            )
        return self.cache[cache_key]


def starrocks_query(sql):
    connection = pymysql.connect(**STARROCKS_CONNECTION)
    try:
        with connection.cursor() as cursor:
            cursor.execute(sql)
            columns = [field[0] for field in cursor.description]
            return [dict(zip(columns, row)) for row in cursor.fetchall()]
    finally:
        connection.close()


def hdfs_path_exists(path):
    jvm = spark._jvm
    hconf = spark._jsc.hadoopConfiguration()
    path_obj = jvm.org.apache.hadoop.fs.Path(ensure_s3a_path(path))
    return path_obj.getFileSystem(hconf).exists(path_obj)


def hdfs_list_files(path):
    jvm = spark._jvm
    hconf = spark._jsc.hadoopConfiguration()
    path_obj = jvm.org.apache.hadoop.fs.Path(ensure_s3a_path(path))
    fs = path_obj.getFileSystem(hconf)
    if not fs.exists(path_obj):
        return []
    result = []
    for status in fs.listStatus(path_obj):
        name = status.getPath().getName()
        if status.isFile() and not name.startswith("_") and not name.startswith("."):
            result.append(str(status.getPath()))
    return sorted(result)


def target_path_for_sha(target_root, sha256):
    return f"{target_root.rstrip('/')}/{sha256}/doc.json"


def progress_output_path(batch_index):
    root = ROW_PROGRESS_ROOT.replace("s3a://", "s3://", 1).rstrip("/")
    return f"{root}/batch-{int(batch_index):08d}-{uuid.uuid4().hex}.jsonl"


def bucket_progress_path(bucket_id):
    return f"{BUCKET_PROGRESS_ROOT.rstrip('/')}/bucket-{int(bucket_id):04d}.json"


def parse_progress_batch_index(path):
    name = str(path).rsplit("/", 1)[-1]
    if not (name.startswith("batch-") and name.endswith(".jsonl")):
        return None
    parts = name[:-len(".jsonl")].split("-")
    if len(parts) < 3:
        return None
    try:
        return int(parts[1])
    except (TypeError, ValueError):
        return None


def load_row_progress_file_infos():
    if not hdfs_path_exists(ROW_PROGRESS_ROOT):
        return []
    files = hdfs_list_files(ROW_PROGRESS_ROOT)
    return [
        {"path": path, "batch_index": parse_progress_batch_index(path)}
        for path in files
    ]


def load_row_progress_df(progress_files=None):
    files = progress_files if progress_files is not None else [item["path"] for item in load_row_progress_file_infos()]
    if not files:
        return None
    return spark.read.schema(ROW_PROGRESS_SCHEMA).json(files).where(F.col("source_sha256").isNotNull() & (F.trim(F.col("source_sha256")) != ""))


def load_row_progress_summary():
    progress_file_infos = load_row_progress_file_infos()
    progress_files = [item["path"] for item in progress_file_infos]
    progress_df = load_row_progress_df(progress_files)
    if progress_df is None:
        return {
            "completed_source_sha256_count": 0,
            "completed_batch_index": 0,
            "last_completed_source_sha256": None,
            "oa_count": 0,
            "non_oa_count": 0,
        }
    if progress_df.first() is None:
        return {
            "completed_source_sha256_count": 0,
            "completed_batch_index": 0,
            "last_completed_source_sha256": None,
            "oa_count": 0,
            "non_oa_count": 0,
        }
    latest_df = (
        progress_df
        .groupBy("source_sha256")
        .agg(
            F.max(F.coalesce(F.col("batch_index"), F.lit(0))).alias("batch_index"),
            F.max(F.coalesce(F.col("access_is_oa"), F.lit("false"))).alias("access_is_oa"),
        )
    )
    summary_row = latest_df.agg(
        F.count("*").alias("completed_source_sha256_count"),
        F.max("batch_index").alias("completed_batch_index"),
        F.max("source_sha256").alias("last_completed_source_sha256"),
        F.sum(F.when(F.col("access_is_oa") == F.lit("true"), F.lit(1)).otherwise(F.lit(0))).alias("oa_count"),
    ).first()
    completed_batch_index = int(summary_row["completed_batch_index"] or 0)
    if completed_batch_index > 0:
        latest_batch_files = [item["path"] for item in progress_file_infos if item["batch_index"] == completed_batch_index]
        latest_batch_df = load_row_progress_df(latest_batch_files)
        latest_sha_row = latest_batch_df.agg(F.max("source_sha256").alias("last_completed_source_sha256")).first() if latest_batch_df is not None else None
        last_completed_source_sha256 = latest_sha_row["last_completed_source_sha256"] if latest_sha_row is not None else None
    else:
        last_completed_source_sha256 = summary_row["last_completed_source_sha256"]
    total_count = int(summary_row["completed_source_sha256_count"] or 0)
    oa_count = int(summary_row["oa_count"] or 0)
    return {
        "completed_source_sha256_count": total_count,
        "completed_batch_index": int(completed_batch_index or 0),
        "last_completed_source_sha256": last_completed_source_sha256,
        "oa_count": oa_count,
        "non_oa_count": int(total_count - oa_count),
    }


def write_row_progress(batch_index, progress_rows):
    completed_at = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    output_path = progress_output_path(batch_index)
    client = ClientCache().get_client(output_path)
    with S3LineWriter(output_path, client=client) as writer:
        for progress_row in progress_rows:
            writer.write(json.dumps({
                "source_sha256": progress_row["source_sha256"],
                "processed_path": progress_row["processed_path"],
                "access_is_oa": progress_row["access_is_oa"],
                "batch_index": int(batch_index),
                "completed_at": completed_at,
            }, ensure_ascii=False, separators=(",", ":")))
    return completed_at


def load_bucket_progress(bucket_id):
    path = bucket_progress_path(bucket_id)
    client = ClientCache().get_client(path)
    try:
        stream = read_s3_object(path, client=client)
    except Exception:
        return {
            "bucket_id": int(bucket_id),
            "last_sort_key": None,
            "last_source_sha256": None,
            "completed_batch_index": 0,
        }
    with stream:
        payload = stream.read()
    text = payload.decode("utf-8") if isinstance(payload, bytes) else str(payload)
    item = json.loads(text)
    return {
        "bucket_id": int(item.get("bucket_id") or bucket_id),
        "last_sort_key": item.get("last_sort_key"),
        "last_source_sha256": item.get("last_source_sha256"),
        "completed_batch_index": int(item.get("completed_batch_index") or 0),
    }


def write_bucket_progress(bucket_id, last_sort_key, last_source_sha256, completed_batch_index):
    path = bucket_progress_path(bucket_id)
    client = ClientCache().get_client(path)
    payload = {
        "bucket_id": int(bucket_id),
        "last_sort_key": None if last_sort_key is None else int(last_sort_key),
        "last_source_sha256": last_source_sha256,
        "completed_batch_index": int(completed_batch_index or 0),
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()),
    }
    put_s3_object(path, json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8"), client=client)


def queue_shard_path(queue_root):
    return f"{queue_root.rstrip('/')}/shard-{uuid.uuid4().hex}.jsonl"


def append_queue_rows(queue_root, rows, rows_per_shard=None):
    if not rows:
        return 0
    rows = list(rows)
    rows_per_shard = None if rows_per_shard is None else max(1, int(rows_per_shard))
    client_cache = ClientCache()
    total_count = 0
    start = 0
    while start < len(rows):
        shard_rows = rows[start:] if rows_per_shard is None else rows[start:start + rows_per_shard]
        output_path = queue_shard_path(queue_root)
        client = client_cache.get_client(output_path)
        lines = [json.dumps(row, ensure_ascii=False, separators=(",", ":")) for row in shard_rows]
        put_s3_object(output_path, ("\n".join(lines) + "\n").encode("utf-8"), client=client)
        total_count += len(shard_rows)
        if rows_per_shard is None:
            break
        start += rows_per_shard
    return total_count


def load_queue_file_rows(s3_path, client=None):
    if client is None:
        client = ClientCache().get_client(s3_path)
    stream = read_s3_object(s3_path, client=client)
    with stream:
        payload = stream.read()
    text = payload.decode("utf-8") if isinstance(payload, bytes) else str(payload)
    shard_rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        shard_rows.append(json.loads(line))
    return shard_rows


def rebalance_queue_file(queue_root, s3_path, shard_rows, rows_per_shard):
    rewritten_rows = append_queue_rows(queue_root, shard_rows, rows_per_shard=rows_per_shard)
    try:
        client = ClientCache().get_client(s3_path)
        delete_s3_object(s3_path, dry_run=False, client=client)
    except Exception:
        pass
    return rewritten_rows


def load_queue_rows(queue_root, limit_rows, preserve_file_boundaries=False, rows_per_shard=None):
    files = hdfs_list_files(queue_root.replace("s3://", "s3a://", 1))
    rows = []
    consumed_files = []
    remainder_rows = []
    bad_files = []
    rebalanced_files = []
    for s3a_path in files:
        if len(rows) >= int(limit_rows):
            break
        s3_path = s3a_path.replace("s3a://", "s3://", 1)
        client = ClientCache().get_client(s3_path)
        try:
            shard_rows = load_queue_file_rows(s3_path, client=client)
        except Exception:
            bad_files.append(s3_path)
            continue
        if preserve_file_boundaries and rows_per_shard and len(shard_rows) > int(rows_per_shard):
            rebalanced_files.append({
                "path": s3_path,
                "row_count": len(shard_rows),
                "rewritten_rows": rebalance_queue_file(queue_root, s3_path, shard_rows, rows_per_shard),
            })
            continue
        remaining_capacity = int(limit_rows) - len(rows)
        if preserve_file_boundaries and rows and len(shard_rows) > remaining_capacity:
            break
        consumed_files.append(s3_path)
        if remaining_capacity > 0:
            rows.extend(shard_rows[:remaining_capacity])
            remainder_rows.extend(shard_rows[remaining_capacity:])
        else:
            remainder_rows.extend(shard_rows)
        if len(rows) >= int(limit_rows):
            break
    return rows, consumed_files, remainder_rows, bad_files, rebalanced_files


def rewrite_queue_rows(queue_root, consumed_files, remaining_rows, bad_files=None, rows_per_shard=None):
    appended_remaining_rows = 0
    if remaining_rows:
        appended_remaining_rows = append_queue_rows(queue_root, remaining_rows, rows_per_shard=rows_per_shard)
    for path in consumed_files:
        try:
            client = ClientCache().get_client(path)
            delete_s3_object(path, dry_run=False, client=client)
        except Exception:
            pass
    for path in (bad_files or []):
        try:
            client = ClientCache().get_client(path)
            delete_s3_object(path, dry_run=False, client=client)
        except Exception:
            pass
    return appended_remaining_rows


def count_queue_files(queue_root):
    return len(hdfs_list_files(queue_root.replace("s3://", "s3a://", 1)))


def format_seconds_compact(total_seconds):
    total_seconds = max(0, int(round(float(total_seconds or 0.0))))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours > 0:
        return f"{hours}h{minutes:02d}m{seconds:02d}s"
    if minutes > 0:
        return f"{minutes}m{seconds:02d}s"
    return f"{seconds}s"


def estimate_remaining_batches_by_queue_files(remaining_queue_files, completed_batches, consumed_queue_files):
    remaining_queue_files = max(0, int(remaining_queue_files or 0))
    completed_batches = max(0, int(completed_batches or 0))
    consumed_queue_files = max(0, int(consumed_queue_files or 0))
    if remaining_queue_files == 0:
        return 0
    if completed_batches == 0 or consumed_queue_files == 0:
        return remaining_queue_files
    avg_queue_files_per_batch = float(consumed_queue_files) / float(completed_batches)
    if avg_queue_files_per_batch <= 0:
        return remaining_queue_files
    return int(math.ceil(float(remaining_queue_files) / avg_queue_files_per_batch))


def append_slow_path_rows(rows):
    return append_queue_rows(SLOW_PATH_QUEUE_ROOT, rows, rows_per_shard=SLOW_PATH_QUEUE_SHARD_ROWS)


def load_slow_path_rows(limit_rows=SLOW_PATH_BATCH_SIZE):
    all_bad_files = []
    all_rebalanced_files = []
    while True:
        rows, consumed_files, remainder_rows, bad_files, rebalanced_files = load_queue_rows(
            SLOW_PATH_QUEUE_ROOT,
            limit_rows,
            preserve_file_boundaries=True,
            rows_per_shard=SLOW_PATH_QUEUE_SHARD_ROWS,
        )
        all_bad_files.extend(bad_files)
        all_rebalanced_files.extend(rebalanced_files)
        if rows or not rebalanced_files:
            return rows, consumed_files, remainder_rows, all_bad_files, all_rebalanced_files


def rewrite_slow_path_rows(consumed_files, remaining_rows, bad_files=None):
    return rewrite_queue_rows(
        SLOW_PATH_QUEUE_ROOT,
        consumed_files,
        remaining_rows,
        bad_files,
        rows_per_shard=SLOW_PATH_QUEUE_SHARD_ROWS,
    )


def append_oversized_path_rows(rows):
    return append_queue_rows(OVERSIZED_PATH_QUEUE_ROOT, rows)


In [ ]:
def parse_content_list_value(value):
    if value is None:
        return None
    if isinstance(value, (list, dict)):
        return value
    if not isinstance(value, str):
        return value
    text = value.strip()
    if not text:
        return value
    try:
        parsed = json.loads(text)
    except (TypeError, ValueError):
        return value
    while isinstance(parsed, str):
        stripped = parsed.strip()
        if not stripped:
            return parsed
        if not (stripped.startswith("[") or stripped.startswith("{") or stripped.startswith('"')):
            return parsed
        try:
            reparsed = json.loads(stripped)
        except (TypeError, ValueError):
            return parsed
        if reparsed == parsed:
            return parsed
        parsed = reparsed
    return parsed


def should_repair_content_list(content_list, raw_content_list):
    if raw_content_list is None:
        return False
    if isinstance(content_list, list):
        return False
    if isinstance(content_list, dict):
        return True
    if isinstance(content_list, str):
        stripped = content_list.strip()
        return stripped.startswith("[") or stripped.startswith("{") or stripped.startswith('"')
    return content_list is None


def normalize_content_list_for_payload(doc):
    raw_content_list = doc.get("content_list")
    content_list = doc.get("content_list")
    if isinstance(content_list, list):
        normalized = []
        for item in content_list:
            if hasattr(item, "asDict"):
                normalized.append(item.asDict(recursive=False))
            else:
                normalized.append(item)
        return normalized
    if should_repair_content_list(content_list, raw_content_list):
        repaired = parse_content_list_value(raw_content_list)
        if repaired is not None:
            return repaired
    if content_list is not None:
        return parse_content_list_value(content_list)
    return content_list


def decode_real_doc(processed_path, client=None):
    raw_text = None
    doc = None
    if "?bytes=" in processed_path:
        base_path, bytes_param = processed_path.split("?bytes=", 1)
        stream = read_s3_object(base_path, bytes_param, client=client)
        with stream:
            raw_value = stream.read()
        if base_path.endswith(".gz"):
            raw_value = gzip.GzipFile(fileobj=io.BytesIO(raw_value)).read()
        elif base_path.endswith(".bz2"):
            raw_value = bz2.BZ2File(io.BytesIO(raw_value)).read()
        raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
    else:
        if processed_path.endswith((".jsonl", ".jsonl.gz", ".jsonl.bz2")):
            row = read_s3_row(processed_path, client=client)
            if row is None:
                raise ValueError(f"read_s3_row returned None for {processed_path}")
            raw_value = row.value
            raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
        else:
            stream = read_s3_object(processed_path, client=client)
            with stream:
                raw_value = stream.read()
            if processed_path.endswith(".gz"):
                raw_value = gzip.GzipFile(fileobj=io.BytesIO(raw_value)).read()
            elif processed_path.endswith(".bz2"):
                raw_value = bz2.BZ2File(io.BytesIO(raw_value)).read()
            raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
    raw_text = raw_text.strip()
    if not raw_text:
        raise ValueError(f"empty content for {processed_path}")
    try:
        doc = json.loads(raw_text)
    except (TypeError, ValueError):
        if "?bytes=" in processed_path:
            raise
        if processed_path.endswith((".jsonl", ".jsonl.gz", ".jsonl.bz2")):
            raise
        row = read_s3_row(processed_path, client=client)
        if row is None:
            raise ValueError(f"read_s3_row returned None for {processed_path}")
        raw_value = row.value
        raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
        raw_text = raw_text.strip()
        if not raw_text:
            raise ValueError(f"empty fallback row content for {processed_path}")
        doc = json.loads(raw_text)
    doc["content_list"] = normalize_content_list_for_payload(doc)
    doc["doc_loc"] = doc.get("doc_loc") or processed_path
    doc["processed_path"] = doc.get("processed_path") or doc["doc_loc"]
    return doc


def extract_doc_sha256_candidates(doc):
    candidates = []
    for key in ("sha256", "doc_id", "id"):
        value = doc.get(key)
        if isinstance(value, str):
            value = value.strip().lower()
            if len(value) == 64 and all(ch in "0123456789abcdef" for ch in value):
                candidates.append(value)
    return candidates


def extract_doc_sha256_candidates_from_text(raw_text):
    text = str(raw_text or "")
    seen = set()
    candidates = []
    for match in re.finditer(r'"(?:sha256|doc_id|id)"\s*:\s*"([0-9a-fA-F]{64})"', text):
        candidate = match.group(1).strip().lower()
        if candidate not in seen:
            seen.add(candidate)
            candidates.append(candidate)
    return candidates


def load_real_docs_for_path(processed_path, expected_sha256s, client=None):
    expected_sha256s = {str(item).strip().lower() for item in expected_sha256s if str(item).strip()}
    if not expected_sha256s:
        return {}
    if "?bytes=" in processed_path:
        doc = decode_real_doc(processed_path, client=client)
        docs_by_sha = {}
        for candidate in extract_doc_sha256_candidates(doc):
            docs_by_sha[candidate] = doc
        if not docs_by_sha:
            fallback_sha = normalize_sha256(next(iter(expected_sha256s)))
            docs_by_sha[fallback_sha] = doc
        return {sha: docs_by_sha[sha] for sha in docs_by_sha if sha in expected_sha256s}

    docs_by_sha = {}
    for row in read_s3_rows(processed_path, use_stream=True, client=client):
        raw_value = row.value
        raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
        raw_text = raw_text.strip()
        if not raw_text:
            continue
        text_candidates = extract_doc_sha256_candidates_from_text(raw_text)
        matched_candidates = [candidate for candidate in text_candidates if candidate in expected_sha256s and candidate not in docs_by_sha]
        if not matched_candidates:
            continue
        try:
            doc = json.loads(raw_text)
        except MemoryError as exc:
            raise MemoryError(
                f"json.loads OOM for processed_path={processed_path}, row_loc={getattr(row, 'loc', None)}, raw_text_len={len(raw_text)}, expected_sha_count={len(expected_sha256s)}"
            ) from exc
        doc["content_list"] = normalize_content_list_for_payload(doc)
        doc["doc_loc"] = doc.get("doc_loc") or row.loc or processed_path
        doc["processed_path"] = doc.get("processed_path") or processed_path
        for candidate in extract_doc_sha256_candidates(doc):
            if candidate in expected_sha256s and candidate not in docs_by_sha:
                docs_by_sha[candidate] = doc
        if len(docs_by_sha) >= len(expected_sha256s):
            break
    return docs_by_sha


def build_output_payload(meta_row, real_doc):
    payload = {field: real_doc.get(field) for field in REAL_DOC_FIELDS}
    for field in ("doi", "title", "author", "language", "abstract", "origin_url", "origin_path", "model_name", "model_version"):
        if meta_row.get(field):
            payload[field] = meta_row[field]
    sha256 = (meta_row.get("source_sha256") or real_doc.get("sha256") or real_doc.get("doc_id") or real_doc.get("id") or "").strip()
    if not sha256:
        raise ValueError(f"missing sha256 for processed_path={meta_row['processed_path']}")
    payload["sha256"] = sha256
    payload["doc_id"] = payload.get("doc_id") or real_doc.get("doc_id") or real_doc.get("id") or sha256
    payload["doc_loc"] = payload.get("doc_loc") or meta_row["processed_path"]
    payload["processed_path"] = meta_row["processed_path"]
    return payload


def normalize_sha256(value):
    normalized = (value or "").strip().lower()
    if len(normalized) != 64 or any(ch not in "0123456789abcdef" for ch in normalized):
        raise ValueError(f"invalid sha256={value!r}")
    return normalized


def build_skipped_existing_result(meta_row, source_sha256, target_path):
    return {
        "processed_path": meta_row["processed_path"],
        "source_sha256": source_sha256,
        "sha256": source_sha256,
        "access_is_oa": meta_row["access_is_oa"],
        "target_path": target_path,
        "status": "skipped_existing",
        "read_seconds": 0.0,
        "write_seconds": 0.0,
    }


def split_existing_meta_rows(meta_rows, force_driver_exists_check=None):
    driver_exists_check = ENABLE_DRIVER_PATH_EXISTENCE_CHECK if force_driver_exists_check is None else bool(force_driver_exists_check)
    if OVERWRITE_EXISTING_DOCS or not driver_exists_check:
        return list(meta_rows), []
    pending_rows = []
    existence_checks = []
    for meta_row in meta_rows:
        source_sha256 = (meta_row.get("source_sha256") or "").strip().lower()
        if not source_sha256:
            pending_rows.append(meta_row)
            continue
        target_root = OA_ROOT if meta_row["access_is_oa"] == "true" else OTHERS_ROOT
        target_path = target_path_for_sha(target_root, source_sha256)
        existence_checks.append((meta_row, source_sha256, target_path))
    if not existence_checks:
        return pending_rows, []
    skipped_rows = []
    max_workers = 32 if RUN_MODE == "prod" else 8
    def check_one(item):
        meta_row, source_sha256, target_path = item
        return meta_row, source_sha256, target_path, hdfs_path_exists(target_path)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for meta_row, source_sha256, target_path, exists_flag in executor.map(check_one, existence_checks):
            if exists_flag:
                skipped_rows.append(build_skipped_existing_result(meta_row, source_sha256, target_path))
            else:
                pending_rows.append(meta_row)
    return pending_rows, skipped_rows


def dedup_meta_rows_by_source_sha(meta_rows):
    dedup = {}
    duplicated_sha256s = set()
    for meta_row in meta_rows:
        source_sha256 = (meta_row.get("source_sha256") or "").strip().lower()
        if not source_sha256:
            continue
        if source_sha256 in dedup:
            duplicated_sha256s.add(source_sha256)
            continue
        dedup[source_sha256] = meta_row
    deduped_rows = [dedup[key] for key in sorted(dedup.keys(), key=lambda sha: dedup[sha]["source_sha256"])]
    return deduped_rows, duplicated_sha256s


def build_partition_batches(meta_rows, target_rows_per_partition=ROWS_PER_PARTITION, target_partition_count=TARGET_BATCH_PARTITIONS, max_partition_count=MAX_BATCH_PARTITIONS):
    target_rows_per_partition = max(1, int(target_rows_per_partition or 1))
    target_partition_count = max(1, int(target_partition_count or 1))
    max_partition_count = max(target_partition_count, int(max_partition_count or target_partition_count))
    if meta_rows:
        target_rows_per_partition = min(
            target_rows_per_partition,
            max(1, int(math.ceil(len(meta_rows) / float(target_partition_count))))
        )
    groups = []
    current_path = None
    current_group = []
    group_sizes = []
    for meta_row in meta_rows:
        processed_path = meta_row["processed_path"]
        if current_path is None or processed_path == current_path:
            current_group.append(meta_row)
            current_path = processed_path
            continue
        groups.append(current_group)
        group_sizes.append(len(current_group))
        current_group = [meta_row]
        current_path = processed_path
    if current_group:
        groups.append(current_group)
        group_sizes.append(len(current_group))

    bucket_count = min(max_partition_count, max(1, target_partition_count))
    buckets = [[] for _ in range(bucket_count)]
    bucket_sizes = [0 for _ in range(bucket_count)]
    for group in groups:
        processed_path = group[0]["processed_path"]
        digest = hashlib.md5(processed_path.encode("utf-8")).hexdigest()
        bucket_idx = int(digest[:8], 16) % bucket_count
        buckets[bucket_idx].extend(group)
        bucket_sizes[bucket_idx] += len(group)
    batches = [bucket for bucket in buckets if bucket]
    non_empty_bucket_sizes = [size for size in bucket_sizes if size > 0]
    return {
        "batches": batches,
        "target_rows_per_partition": target_rows_per_partition,
        "max_bucket_size": max(non_empty_bucket_sizes) if non_empty_bucket_sizes else 0,
        "min_bucket_size": min(non_empty_bucket_sizes) if non_empty_bucket_sizes else 0,
    }


def resolve_partition_threading(partition_count, forced_read_threads=None, forced_write_threads=None):
    partition_count = max(1, int(partition_count or 1))
    if forced_read_threads is not None:
        read_threads = max(1, int(forced_read_threads))
    else:
        read_threads = max(1, int(math.ceil(TARGET_GLOBAL_READ_CONCURRENCY / float(partition_count))))
    if forced_write_threads is not None:
        write_threads = max(1, int(forced_write_threads))
    else:
        write_threads = max(1, int(math.ceil(TARGET_GLOBAL_WRITE_CONCURRENCY / float(partition_count))))
    return {
        "read_threads": read_threads,
        "write_threads": write_threads,
    }


def clamp_thread_plan(thread_plan, row_count, max_read_threads=None, max_write_threads=None):
    row_count = max(1, int(row_count or 1))
    max_read_threads = max(1, int(max_read_threads or MAX_S3_CONNECTIONS))
    max_write_threads = max(1, int(max_write_threads or MAX_S3_CONNECTIONS))
    read_threads = max(1, int((thread_plan or {}).get("read_threads") or 1))
    write_threads = max(1, int((thread_plan or {}).get("write_threads") or 1))
    return {
        "read_threads": min(read_threads, row_count, max_read_threads),
        "write_threads": min(write_threads, row_count, max_write_threads),
    }


def process_meta_partition_bundle(item):
    rows, thread_plan = item
    return process_meta_partition(rows, thread_plan)


def process_meta_partition(rows, thread_plan=None):
    client_cache = ClientCache()
    read_threads = int((thread_plan or {}).get("read_threads") or 1)
    write_threads = int((thread_plan or {}).get("write_threads") or 1)
    def write_payload(meta_row, real_doc, read_seconds):
        source_sha256 = (meta_row.get("source_sha256") or "").strip().lower()
        target_root = OA_ROOT if meta_row["access_is_oa"] == "true" else OTHERS_ROOT
        payload = build_output_payload(meta_row, real_doc)
        payload["sha256"] = normalize_sha256(payload["sha256"])
        target_path = target_path_for_sha(target_root, payload["sha256"])
        client = client_cache.get_client(target_path)
        write_started_at = time.perf_counter()
        try:
            put_s3_object(
                target_path,
                json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"),
                client=client,
            )
        except Exception as exc:
            raise RuntimeError(
                f"put_s3_object failed: processed_path={meta_row['processed_path']}, "
                f"sha256={payload['sha256']}, target_path={target_path}, error={type(exc).__name__}: {exc}"
            ) from exc
        write_seconds = time.perf_counter() - write_started_at
        return {
            "processed_path": meta_row["processed_path"],
            "source_sha256": payload["sha256"],
            "sha256": payload["sha256"],
            "access_is_oa": meta_row["access_is_oa"],
            "target_path": target_path,
            "status": "written",
            "read_seconds": read_seconds,
            "write_seconds": write_seconds,
            "total_seconds": read_seconds + write_seconds,
        }


    def read_meta_row(meta_row):
        read_client = client_cache.get_client(meta_row["processed_path"])
        read_started_at = time.perf_counter()
        try:
            real_doc = decode_real_doc(meta_row["processed_path"], client=read_client)
        except Exception as exc:
            byte_len = parse_bytes_length_from_processed_path(meta_row.get("processed_path"))
            raise RuntimeError(
                f"decode_real_doc failed: processed_path={meta_row.get('processed_path')}, "
                f"source_sha256={meta_row.get('source_sha256')}, byte_length={byte_len}, "
                f"error={type(exc).__name__}: {exc}"
            ) from exc
        read_seconds = time.perf_counter() - read_started_at
        return meta_row, real_doc, read_seconds


    def scan_path_group(group_item):
        processed_path, path_rows = group_item
        source_sha256s = [(row.get("source_sha256") or "").strip().lower() for row in path_rows]
        read_client = client_cache.get_client(processed_path)
        read_started_at = time.perf_counter()
        docs_by_sha = load_real_docs_for_path(processed_path, source_sha256s, client=read_client)
        read_seconds = time.perf_counter() - read_started_at
        missing_sha256s = [sha for sha in source_sha256s if sha not in docs_by_sha]
        if missing_sha256s:
            raise RuntimeError(
                f"missing docs after scanning processed_path={processed_path}, missing_count={len(missing_sha256s)}, "
                f"sample_missing={missing_sha256s[:5]}"
            )
        per_row_read_seconds = read_seconds / max(1, len(path_rows))
        return [
            (meta_row, docs_by_sha[(meta_row.get("source_sha256") or "").strip().lower()], per_row_read_seconds)
            for meta_row in path_rows
        ]

    rows = list(rows)
    partition_started_at = time.perf_counter()
    result_rows = []
    grouped_rows = defaultdict(list)
    for meta_row in rows:
        grouped_rows[meta_row["processed_path"]].append(meta_row)
    direct_rows = []
    grouped_scan_groups = []
    for processed_path, path_rows in grouped_rows.items():
        if should_scan_processed_path_by_sha(processed_path):
            grouped_scan_groups.append((processed_path, path_rows))
        elif ("?bytes=" in processed_path) or (len(path_rows) == 1):
            direct_rows.extend(path_rows)
        else:
            grouped_scan_groups.append((processed_path, path_rows))
    if direct_rows:
        read_futures = []
        write_futures = []
        with ThreadPoolExecutor(max_workers=read_threads) as read_executor, ThreadPoolExecutor(max_workers=write_threads) as write_executor:
            for meta_row in direct_rows:
                read_futures.append(read_executor.submit(read_meta_row, meta_row))
            for future in as_completed(read_futures):
                meta_row, real_doc, read_seconds = future.result()
                write_futures.append(write_executor.submit(write_payload, meta_row, real_doc, read_seconds))
            for future in as_completed(write_futures):
                result_rows.append(future.result())
    if grouped_scan_groups:
        scan_futures = []
        write_futures = []
        with ThreadPoolExecutor(max_workers=read_threads) as read_executor, ThreadPoolExecutor(max_workers=write_threads) as write_executor:
            for group_item in grouped_scan_groups:
                scan_futures.append(read_executor.submit(scan_path_group, group_item))
            for future in as_completed(scan_futures):
                for meta_row, real_doc, per_row_read_seconds in future.result():
                    write_futures.append(write_executor.submit(write_payload, meta_row, real_doc, per_row_read_seconds))
            for future in as_completed(write_futures):
                result_rows.append(future.result())
    partition_elapsed = time.perf_counter() - partition_started_at
    if result_rows:
        pass
    for row in result_rows:
        yield row


In [ ]:


def fetch_meta_batch(bucket_id, last_source_sha256=None, limit_rows=BATCH_SIZE):
    fetch_started_at = time.perf_counter()
    bucket_clause = f"AND abs(crc32(lower(trim(access_xinghe_repository_sha256)))) % {int(SHA_BUCKET_COUNT)} = {int(bucket_id)}"
    if RUN_MODE == "test" and last_source_sha256 is None:
        base_sql = SQL_TEMPLATES["fetch_meta_base"].format(
            starrocks_table=STARROCKS_TABLE,
            bucket_clause=bucket_clause,
            last_clause="",
            limit_rows=int(TEST_ROW_LIMIT),
        )
        query_started_at = time.perf_counter()
        rows = starrocks_query(base_sql)
        base_query_elapsed = time.perf_counter() - query_started_at
        extra_query_elapsed = 0.0
        if TEST_TARGET_SHA256S:
            quoted_sha256s = ", ".join(quote_sql(item.strip().lower()) for item in TEST_TARGET_SHA256S)
            extra_sql = SQL_TEMPLATES["fetch_meta_test_target_sha256s"].format(
                starrocks_table=STARROCKS_TABLE,
                bucket_clause=bucket_clause,
                quoted_sha256s=quoted_sha256s,
            )
            extra_query_started_at = time.perf_counter()
            rows.extend(starrocks_query(extra_sql))
            extra_query_elapsed = time.perf_counter() - extra_query_started_at
        dedup = {}
        for row in rows:
            source_sha256 = (row.get("source_sha256") or "").strip().lower()
            processed_path = (row.get("processed_path") or "").strip()
            if source_sha256 and processed_path:
                dedup[source_sha256] = row
        rows = [dedup[key] for key in sorted(dedup.keys())]
    else:
        last_clause = ""
        if last_source_sha256:
            last_clause = f"AND lower(trim(access_xinghe_repository_sha256)) > {quote_sql(last_source_sha256)}"
        sql = SQL_TEMPLATES["fetch_meta_base"].format(
            starrocks_table=STARROCKS_TABLE,
            bucket_clause=bucket_clause,
            last_clause=last_clause,
            limit_rows=int(limit_rows),
        )
        query_started_at = time.perf_counter()
        rows = starrocks_query(sql)
        base_query_elapsed = time.perf_counter() - query_started_at
        extra_query_elapsed = 0.0
    normalized = []
    for row in rows:
        processed_path = (row.get("processed_path") or "").strip()
        source_sha256 = (row.get("source_sha256") or "").strip().lower()
        if not processed_path:
            continue
        if not source_sha256:
            continue
        normalized.append({
            "metadata_type": row.get("metadata_type"),
            "doi": row.get("doi"),
            "isbn13": row.get("isbn13"),
            "title": row.get("title"),
            "author": row.get("author"),
            "language": row.get("language"),
            "origin_url": row.get("origin_url"),
            "origin_path": row.get("origin_path"),
            "processed_path": processed_path,
            "source_sha256": source_sha256,
            "model_name": row.get("model_name"),
            "model_version": row.get("model_version"),
            "access_is_oa": normalize_bool_text(row.get("access_is_oa")),
            "abstract": row.get("abstract"),
        })
    normalized.sort(key=lambda item: item["source_sha256"])
    fetch_elapsed = time.perf_counter() - fetch_started_at
    print(
        f"  拉取批次完成: rows={len(normalized)}, base_query={base_query_elapsed:.2f}s, "
        f"extra_query={extra_query_elapsed:.2f}s, total={fetch_elapsed:.2f}s, "
        f"bucket_id={bucket_id}, last_source_sha256={last_source_sha256}"
    )
    return normalized


def count_pending_rows(bucket_id, last_source_sha256=None):
    if RUN_MODE == "test":
        first_batch = fetch_meta_batch(bucket_id, last_source_sha256, limit_rows=BATCH_SIZE)
        return len(first_batch)
    last_clause = ""
    if last_source_sha256:
        last_clause = f"AND lower(trim(access_xinghe_repository_sha256)) > {quote_sql(last_source_sha256)}"
    bucket_clause = f"AND abs(crc32(lower(trim(access_xinghe_repository_sha256)))) % {int(SHA_BUCKET_COUNT)} = {int(bucket_id)}"
    sql = SQL_TEMPLATES["count_pending_rows"].format(
        starrocks_table=STARROCKS_TABLE,
        bucket_clause=bucket_clause,
        last_clause=last_clause,
    )
    rows = starrocks_query(sql)
    return int(rows[0]["total_count"] or 0) if rows else 0


def write_batch(batch_index, meta_rows, enable_slow_path_split=True, target_partitions=TARGET_BATCH_PARTITIONS, rows_per_partition=ROWS_PER_PARTITION, forced_read_threads=None, forced_write_threads=None, force_driver_exists_check=None):
    driver_exists_check = ENABLE_DRIVER_PATH_EXISTENCE_CHECK if force_driver_exists_check is None else bool(force_driver_exists_check)
    print(
        f"  批前过滤开始: batch_index={batch_index}, metadata_rows={len(meta_rows)}, "
        f"driver_exists_check={driver_exists_check}"
    )
    dedup_started_at = time.perf_counter()
    deduped_meta_rows, duplicated_sha256s = dedup_meta_rows_by_source_sha(meta_rows)
    dedup_elapsed = time.perf_counter() - dedup_started_at
    if duplicated_sha256s:
        print(
            f"  批内去重完成: before={len(meta_rows)}, after={len(deduped_meta_rows)}, "
            f"duplicated_sha256s={len(duplicated_sha256s)}, 耗时={dedup_elapsed:.2f}s"
        )
    meta_rows = deduped_meta_rows
    split_started_at = time.perf_counter()
    pending_meta_rows, skipped_existing_rows = split_existing_meta_rows(meta_rows, force_driver_exists_check=driver_exists_check)
    split_elapsed = time.perf_counter() - split_started_at
    print(
        f"  批前过滤完成: pending_rows={len(pending_meta_rows)}, skipped_existing_rows={len(skipped_existing_rows)}, "
        f"耗时={split_elapsed:.2f}s"
    )
    oversized_path_rows = []
    oversized_path_candidates = [
        row for row in pending_meta_rows
        if (parse_bytes_length_from_processed_path(row.get("processed_path")) or 0) >= int(OVERSIZED_BYTES_THRESHOLD)
    ]
    if oversized_path_candidates:
        oversized_path_rows = oversized_path_candidates
        oversized_sha_set = {row["source_sha256"] for row in oversized_path_rows}
        pending_meta_rows = [row for row in pending_meta_rows if row["source_sha256"] not in oversized_sha_set]
        appended_oversized_rows = append_oversized_path_rows(oversized_path_rows)
        print(
            f"  超大文档拆分: oversized_rows={len(oversized_path_rows)}, appended_to_queue={appended_oversized_rows}, "
            f"remaining_rows={len(pending_meta_rows)}"
        )
    slow_path_rows = []
    if enable_slow_path_split:
        def is_large_bytes_path(processed_path):
            if "?bytes=" not in processed_path:
                return False
            try:
                _, bytes_part = processed_path.split("?bytes=", 1)
                _, length_str = bytes_part.split(",", 1)
                return int(length_str) >= int(SLOW_PATH_BYTES_THRESHOLD)
            except Exception:
                return False
        slow_path_candidates = [
            row for row in pending_meta_rows
            if ((("?bytes=" not in row["processed_path"]) and row["processed_path"].endswith((".jsonl", ".jsonl.gz", ".jsonl.bz2"))) or is_large_bytes_path(row["processed_path"]))
        ]
        if slow_path_candidates:
            slow_path_rows = slow_path_candidates
            slow_sha_set = {row["source_sha256"] for row in slow_path_rows}
            pending_meta_rows = [row for row in pending_meta_rows if row["source_sha256"] not in slow_sha_set]
            appended_slow_rows = append_slow_path_rows(slow_path_rows)
            print(
                f"  慢路径拆分: slow_path_rows={len(slow_path_rows)}, appended_to_queue={appended_slow_rows}, "
                f"remaining_fast_rows={len(pending_meta_rows)}"
            )
    execution_mode = "skip"
    pending_path_type_summary = summarize_processed_path_types(pending_meta_rows)
    if pending_path_type_summary:
        print(f"  待写路径类型分布: {pending_path_type_summary}")
    if pending_meta_rows:
        pending_meta_rows = sorted(pending_meta_rows, key=lambda row: (row["processed_path"], row["source_sha256"]))
        pending_row_count = len(pending_meta_rows)
        unique_processed_path_count = len({row["processed_path"] for row in pending_meta_rows})
        if pending_row_count <= int(DRIVER_WRITE_MAX_PENDING_ROWS):
            partition_count = 1
            thread_plan = clamp_thread_plan(
                resolve_partition_threading(partition_count, forced_read_threads, forced_write_threads),
                pending_row_count,
                DRIVER_WRITE_MAX_READ_THREADS,
                DRIVER_WRITE_MAX_WRITE_THREADS,
            )
            print(
                f"  Driver 直写开始: pending_rows={pending_row_count}, unique_processed_paths={unique_processed_path_count}, "
                f"read_threads={thread_plan['read_threads']}, write_threads={thread_plan['write_threads']}"
            )
            spark_started_at = time.perf_counter()
            result_rows = list(process_meta_partition(pending_meta_rows, thread_plan))
            spark_elapsed = time.perf_counter() - spark_started_at
            execution_mode = "driver"
            print(f"  Driver 直写完成: result_rows={len(result_rows)}, 耗时={spark_elapsed:.2f}s")
        else:
            partition_plan = build_partition_batches(pending_meta_rows, rows_per_partition, target_partitions, MAX_BATCH_PARTITIONS)
            partition_batches = partition_plan["batches"]
            partition_count = len(partition_batches)
            thread_plan = clamp_thread_plan(
                resolve_partition_threading(partition_count, forced_read_threads, forced_write_threads),
                partition_plan["max_bucket_size"] or pending_row_count,
            )
            print(
                f"  Spark 写入开始: pending_rows={pending_row_count}, partition_count={partition_count}, "
                f"rows_per_partition={partition_plan['target_rows_per_partition']}, unique_processed_paths={unique_processed_path_count}, "
                f"bucket_min={partition_plan['min_bucket_size']}, bucket_max={partition_plan['max_bucket_size']}, "
                f"read_threads={thread_plan['read_threads']}, "
                f"write_threads={thread_plan['write_threads']}"
            )
            spark_started_at = time.perf_counter()
            result_rows = (
                spark.sparkContext.parallelize([(batch, thread_plan) for batch in partition_batches], partition_count)
                .flatMap(process_meta_partition_bundle)
                .collect()
            )
            spark_elapsed = time.perf_counter() - spark_started_at
            execution_mode = "spark"
            print(f"  Spark 写入完成: result_rows={len(result_rows)}, 耗时={spark_elapsed:.2f}s")
    else:
        partition_count = 0
        result_rows = []
        spark_elapsed = 0.0
        print("  Spark 写入跳过: pending_rows=0")
    result_rows.extend(skipped_existing_rows)
    processed_sha256s = [row["source_sha256"] for row in result_rows]
    progress_rows = [
        {
            "source_sha256": row["source_sha256"],
            "processed_path": row["processed_path"],
            "access_is_oa": row["access_is_oa"],
        }
        for row in result_rows
    ]
    written_count = sum(1 for row in result_rows if row["status"] == "written")
    skipped_existing_count = sum(1 for row in result_rows if row["status"] == "skipped_existing")
    total_read_seconds = sum(float(row.get("read_seconds") or 0.0) for row in result_rows)
    total_write_seconds = sum(float(row.get("write_seconds") or 0.0) for row in result_rows)
    slowest_result_rows = summarize_result_rows(result_rows, top_n=5)
    if slowest_result_rows:
        print("  最慢写入样本:")
        for idx, item in enumerate(slowest_result_rows, start=1):
            print(
                f"    {idx}. sha256={item['source_sha256']}, path_type={item['path_type']}, "
                f"read={item['read_seconds']:.2f}s, write={item['write_seconds']:.2f}s, total={item['total_seconds']:.2f}s, "
                f"processed_path={item['processed_path']}"
            )
    print(f"  Checkpoint 写入开始: source_sha256s={len(progress_rows)}")
    checkpoint_started_at = time.perf_counter()
    completed_at = write_row_progress(batch_index, progress_rows)
    checkpoint_elapsed = time.perf_counter() - checkpoint_started_at
    print(f"  Checkpoint 写入完成: source_sha256s={len(progress_rows)}, 耗时={checkpoint_elapsed:.2f}s")
    return {
        "written_count": written_count,
        "skipped_existing_count": skipped_existing_count,
        "processed_count": len(processed_sha256s),
        "execution_mode": execution_mode,
        "partition_count": partition_count,
        "split_elapsed": split_elapsed,
        "spark_elapsed": spark_elapsed,
        "checkpoint_elapsed": checkpoint_elapsed,
        "total_read_seconds": total_read_seconds,
        "total_write_seconds": total_write_seconds,
        "pending_path_type_summary": pending_path_type_summary,
        "slowest_result_rows": slowest_result_rows,
        "slow_path_count": len(slow_path_rows),
        "oversized_path_count": len(oversized_path_rows),
        "last_sort_key": None,
        "completed_at": completed_at,
        "last_source_sha256": meta_rows[-1]["source_sha256"] if meta_rows else None,
    }


In [ ]:
progress_summary = load_row_progress_summary()
completed_source_sha256_count = progress_summary["completed_source_sha256_count"]
bucket_progresses = [load_bucket_progress(bucket_id) for bucket_id in range(SHA_BUCKET_COUNT)]
completed_batch_index = max(
    int(progress_summary.get("completed_batch_index") or 0),
    max((item["completed_batch_index"] for item in bucket_progresses), default=0),
)
pending_row_count = 0
pending_batch_count = 0
preview_bucket_id = None
preview_rows = []
for item in bucket_progresses:
    bucket_id = item["bucket_id"]
    last_sort_key = item["last_sort_key"]
    last_source_sha256 = item["last_source_sha256"]
    bucket_pending_count = count_pending_rows(bucket_id, last_source_sha256)
    pending_row_count += bucket_pending_count
    if bucket_pending_count > 0:
        pending_batch_count += (bucket_pending_count + BATCH_SIZE - 1) // BATCH_SIZE
        if preview_bucket_id is None:
            preview_bucket_id = bucket_id
            preview_rows = fetch_meta_batch(bucket_id, last_source_sha256, limit_rows=PREVIEW_ROWS)
estimated_total_batch_count = completed_batch_index + pending_batch_count
print(f"已完成 source_sha256 数 = {completed_source_sha256_count}")
print(f"已完成全局批次号 = {completed_batch_index}")
print(f"待处理 metadata 行数 = {pending_row_count}")
print(f"预计待处理批次数(不含慢路径) = {pending_batch_count}")
print(f"预计续跑后的全局总批次数(不含慢路径) = {estimated_total_batch_count}")
print(f"预览 bucket_id = {preview_bucket_id}")
print(f"下一批预览条数 = {len(preview_rows)}")
for row in preview_rows:
    print(row["processed_path"], row["source_sha256"], row["access_is_oa"])


In [ ]:
global_batch_index = completed_batch_index
total_written_count = 0
total_skipped_existing_count = 0
total_processed_count = 0

if RUN_MAIN_FLOW:
    for bucket_id in range(SHA_BUCKET_COUNT):
        bucket_progress = load_bucket_progress(bucket_id)
        cursor_sort_key = bucket_progress["last_sort_key"]
        cursor_sha256 = bucket_progress["last_source_sha256"]
        while True:
            batch_rows = fetch_meta_batch(bucket_id, cursor_sha256, limit_rows=BATCH_SIZE)
            if not batch_rows:
                break
            global_batch_index += 1
            print(f"批次 {global_batch_index}: bucket_id={bucket_id}, 本批 metadata 行数 = {len(batch_rows)}")
            started_at = time.perf_counter()
            batch_result = write_batch(global_batch_index, batch_rows)
            elapsed = time.perf_counter() - started_at
            total_written_count += batch_result["written_count"]
            total_skipped_existing_count += batch_result["skipped_existing_count"]
            total_processed_count += batch_result["processed_count"]
            cursor_sort_key = batch_result["last_sort_key"]
            cursor_sha256 = batch_result["last_source_sha256"]
            write_bucket_progress(bucket_id, cursor_sort_key, cursor_sha256, global_batch_index)
            avg_read_seconds = (batch_result["total_read_seconds"] / batch_result["written_count"]) if batch_result["written_count"] else 0.0
            avg_write_seconds = (batch_result["total_write_seconds"] / batch_result["written_count"]) if batch_result["written_count"] else 0.0
            effective_read_concurrency = (batch_result["total_read_seconds"] / elapsed) if elapsed > 0 else 0.0
            effective_write_concurrency = (batch_result["total_write_seconds"] / elapsed) if elapsed > 0 else 0.0
            print(
                f"  执行模式 = {batch_result['execution_mode']}，实际分区数 = {batch_result['partition_count']}，写入 = {batch_result['written_count']}，已存在跳过 = {batch_result['skipped_existing_count']}，"
                f"checkpoint = {batch_result['processed_count']}，read耗时汇总 = {batch_result['total_read_seconds']:.2f}s，"
                f"write耗时汇总 = {batch_result['total_write_seconds']:.2f}s，平均单条read = {avg_read_seconds:.3f}s，"
                f"平均单条write = {avg_write_seconds:.3f}s，等效read并发 = {effective_read_concurrency:.2f}，"
                f"等效write并发 = {effective_write_concurrency:.2f}，completed_at = {batch_result['completed_at']}，批次总耗时 = {elapsed:.2f}s"
            )
            if RUN_MODE == "test":
                break
        if RUN_MODE == "test":
            break

print(f"主流程完成，新增写入 = {total_written_count}")
print(f"主流程已存在跳过 = {total_skipped_existing_count}")
print(f"主流程新增 checkpoint 行数 = {total_processed_count}")
print(f"最新全局批次号 = {global_batch_index}")

if RUN_SLOW_PATH:
    slow_batch_index = global_batch_index
    slow_completed_batches = 0
    slow_total_input_rows = 0
    slow_total_written_rows = 0
    slow_total_elapsed_seconds = 0.0
    slow_total_consumed_files = 0
    while True:
        slow_rows, consumed_files, remainder_rows, bad_files, rebalanced_files = load_slow_path_rows(SLOW_PATH_BATCH_SIZE)
        if not slow_rows:
            print("慢路径队列为空")
            break
        if rebalanced_files:
            print(f"慢路径队列重分片: files={len(rebalanced_files)}")
        slow_batch_index += 1
        print(f"慢路径批次 {slow_batch_index}: rows={len(slow_rows)}")
        sample_slow_sha256 = next((str(row.get("source_sha256") or "").strip().lower() for row in slow_rows if str(row.get("source_sha256") or "").strip()), None)
        if sample_slow_sha256:
            print(f"慢路径批次样例 sha256: {sample_slow_sha256}")
        slow_started_at = time.perf_counter()
        slow_result = write_batch(
            slow_batch_index,
            slow_rows,
            enable_slow_path_split=False,
            target_partitions=SLOW_PATH_TARGET_PARTITIONS,
            rows_per_partition=max(1, int(math.ceil(len(slow_rows) / float(SLOW_PATH_TARGET_PARTITIONS)))),
            forced_read_threads=SLOW_PATH_READ_THREADS,
            forced_write_threads=SLOW_PATH_WRITE_THREADS,
            force_driver_exists_check=SLOW_PATH_FORCE_DRIVER_EXISTS_CHECK,
        )
        slow_elapsed = time.perf_counter() - slow_started_at
        total_written_count += slow_result["written_count"]
        total_skipped_existing_count += slow_result["skipped_existing_count"]
        total_processed_count += slow_result["processed_count"]
        appended_remainder_rows = rewrite_slow_path_rows(consumed_files, remainder_rows, bad_files)
        remaining_slow_queue_files = count_queue_files(SLOW_PATH_QUEUE_ROOT)
        slow_completed_batches += 1
        slow_total_input_rows += len(slow_rows)
        slow_total_written_rows += slow_result["written_count"]
        slow_total_elapsed_seconds += slow_elapsed
        slow_total_consumed_files += len(consumed_files)
        avg_slow_batch_rows = float(slow_total_input_rows) / float(slow_completed_batches)
        avg_slow_batch_written = float(slow_total_written_rows) / float(slow_completed_batches)
        avg_slow_batch_elapsed = float(slow_total_elapsed_seconds) / float(slow_completed_batches)
        avg_slow_queue_files_per_batch = float(slow_total_consumed_files) / float(slow_completed_batches)
        estimated_remaining_batches = estimate_remaining_batches_by_queue_files(
            remaining_slow_queue_files,
            slow_completed_batches,
            slow_total_consumed_files,
        )
        estimated_remaining_eta_seconds = estimated_remaining_batches * avg_slow_batch_elapsed
        print(
            f"慢路径批次完成: execution_mode={slow_result['execution_mode']}, written={slow_result['written_count']}, skipped_existing={slow_result['skipped_existing_count']}, "
            f"remainder_rows={len(remainder_rows)}, appended_remainder_rows={appended_remainder_rows}, bad_files={len(bad_files)}, elapsed={slow_elapsed:.2f}s"
        )
        print(
            f"慢路径剩余估算: queue_files={remaining_slow_queue_files}, estimated_batches={estimated_remaining_batches}, "
            f"eta={format_seconds_compact(estimated_remaining_eta_seconds)}, avg_batch_rows={avg_slow_batch_rows:.1f}, "
            f"avg_written={avg_slow_batch_written:.1f}, avg_elapsed={avg_slow_batch_elapsed:.2f}s, "
            f"avg_queue_files_per_batch={avg_slow_queue_files_per_batch:.2f}"
        )

print(f"超大文档队列 shard 数量 = {count_queue_files(OVERSIZED_PATH_QUEUE_ROOT)}")


In [ ]:
final_progress_summary = load_row_progress_summary()
total_count = int(final_progress_summary["completed_source_sha256_count"])
oa_count = int(final_progress_summary["oa_count"])

summary_payload = {
    "total_count": int(total_count),
    "oa_count": int(oa_count),
    "non_oa_count": int(total_count - oa_count),
}
put_s3_object(SUMMARY_PATH, json.dumps(summary_payload, ensure_ascii=False, indent=2).encode("utf-8"))
print(summary_payload)
